In [1]:
# ============================================================
# CELL 1 — PROJECT SETUP AND LOAD 1-MINUTE SOLETE DATASET
# ============================================================

from pathlib import Path
import sys

import pandas as pd


# ------------------------------------------------------------
# PROJECT SETUP
# ------------------------------------------------------------

# The notebook is stored inside:
#
#     Project/notebooks/
#
# Therefore, the project root is one directory above
# the current notebook location.
PROJECT_ROOT = Path.cwd().parent

# Add the project root to Python's import path so that
# reusable modules inside src/ can be imported.
sys.path.append(str(PROJECT_ROOT))


# ------------------------------------------------------------
# IMPORT OUR REUSABLE SOLETE LOADER
# ------------------------------------------------------------

# This loader is resolution-independent.
# The selected HDF5 file determines whether we load
# 60-minute, 5-minute, 1-minute, or 1-second data.
from src.data_loader import load_solete


# ------------------------------------------------------------
# LOAD THE 1-MINUTE DATASET
# ------------------------------------------------------------

dataset_path = Path(
    "../solete_dataset/SOLETE_Pombo_1min.h5"
)

df = load_solete(dataset_path)


# ------------------------------------------------------------
# BASIC CONFIRMATION
# ------------------------------------------------------------

print("Dataset loaded successfully.")

print("\nDataset shape:")
print(df.shape)

print("\nFirst timestamp:")
print(df.index.min())

print("\nLast timestamp:")
print(df.index.max())

print("\nFirst 5 observations:")
df.head()

Dataset loaded successfully.

Dataset shape:
(658081, 11)

First timestamp:
2018-06-01 00:00:00+00:00

Last timestamp:
2019-09-01 00:00:00+00:00

First 5 observations:


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg]
Timestamp,,,,,,,,,,,
2018-06-01 00:00:00+00:00,14.600000,0.7,1.311864,108.135593,0.0,0.0,0.038733,0.0,1017.599976,0.0,0.0
2018-06-01 00:01:00+00:00,14.600000,0.7,1.548333,115.033333,0.0,0.0,0.000033,0.0,1017.599976,0.0,0.0
2018-06-01 00:02:00+00:00,14.596667,0.7,1.365000,104.683333,0.0,0.0,0.029767,0.0,1017.536658,0.0,0.0
2018-06-01 00:03:00+00:00,14.510000,0.7,0.980000,111.400000,0.0,0.0,0.000000,0.0,1017.463342,0.0,0.0
2018-06-01 00:04:00+00:00,14.500000,0.7,1.263333,118.866667,0.0,0.0,0.166050,0.0,1017.599976,0.0,0.0


In [2]:
# ============================================================
# CELL 2 — STRUCTURAL INTEGRITY CHECKS
# ============================================================

# This cell performs the essential structural checks for
# the 1-minute SOLETE dataset:
#
#   1. Dataset dimensions
#   2. Date range
#   3. Missing values
#   4. Duplicate timestamps
#   5. Chronological ordering
#   6. Sampling interval
#
# We already performed these checks in detail for the
# 60-minute and 5-minute datasets, so here we keep them
# together in one compact diagnostic cell.


# ------------------------------------------------------------
# 1. DATASET SIZE
# ------------------------------------------------------------

print("Dataset shape:")
print(df.shape)


# ------------------------------------------------------------
# 2. TEMPORAL COVERAGE
# ------------------------------------------------------------

print("\nFirst timestamp:")
print(df.index.min())

print("\nLast timestamp:")
print(df.index.max())


# ------------------------------------------------------------
# 3. MISSING VALUES
# ------------------------------------------------------------

missing_values = df.isna().sum()

print("\nMissing values by column:")
print(missing_values)

print("\nTotal missing values:")
print(missing_values.sum())


# ------------------------------------------------------------
# 4. DUPLICATE TIMESTAMPS
# ------------------------------------------------------------

duplicate_timestamps = df.index.duplicated().sum()

print("\nDuplicate timestamps:")
print(duplicate_timestamps)

print("\nAre all timestamps unique?")
print(df.index.is_unique)


# ------------------------------------------------------------
# 5. CHRONOLOGICAL ORDER
# ------------------------------------------------------------

print("\nAre timestamps in chronological order?")
print(df.index.is_monotonic_increasing)


# ------------------------------------------------------------
# 6. SAMPLING INTERVAL
# ------------------------------------------------------------

time_differences = df.index.to_series().diff()

print("\nMost common time intervals:")
print(
    time_differences
    .value_counts()
    .head(5)
)

# For the 1-minute dataset, every consecutive timestamp
# should differ by exactly one minute.
unexpected_intervals = (
    time_differences.iloc[1:]
    != pd.Timedelta(minutes=1)
).sum()

print("\nNumber of unexpected time intervals:")
print(unexpected_intervals)

Dataset shape:
(658081, 11)

First timestamp:
2018-06-01 00:00:00+00:00

Last timestamp:
2019-09-01 00:00:00+00:00

Missing values by column:
TEMPERATURE[degC]    0
HUMIDITY[%]          0
WIND_SPEED[m1s]      0
WIND_DIR[deg]        0
GHI[kW1m2]           0
POA Irr[kW1m2]       0
P_Gaia[kW]           0
P_Solar[kW]          0
Pressure[mbar]       0
Azimuth[deg]         0
Elevation[deg]       0
dtype: int64

Total missing values:
0

Duplicate timestamps:
0

Are all timestamps unique?
True

Are timestamps in chronological order?
True

Most common time intervals:
Timestamp
0 days 00:01:00    658080
Name: count, dtype: int64

Number of unexpected time intervals:
0


In [3]:
# ============================================================
# CELL 3 — STATISTICAL SUMMARY AND RANGE SCREENING
# ============================================================

# This cell combines two tasks:
#
#   1. Basic descriptive statistics
#   2. Screening for the anomalies that we already found
#      in the 60-minute and 5-minute datasets
#
# We are NOT modifying any values.


# ------------------------------------------------------------
# 1. DESCRIPTIVE STATISTICS
# ------------------------------------------------------------

summary_statistics = df.describe().T

print("Statistical summary:")
display(summary_statistics)


# ------------------------------------------------------------
# 2. KEY RANGE / VALIDITY CHECKS
# ------------------------------------------------------------

checks = {
    "Temperature < -50°C": (
        df["TEMPERATURE[degC]"] < -50
    ).sum(),

    "Temperature > 60°C": (
        df["TEMPERATURE[degC]"] > 60
    ).sum(),

    "Humidity < 0": (
        df["HUMIDITY[%]"] < 0
    ).sum(),

    "Wind speed < 0": (
        df["WIND_SPEED[m1s]"] < 0
    ).sum(),

    "Wind direction < 0°": (
        df["WIND_DIR[deg]"] < 0
    ).sum(),

    "Wind direction > 360°": (
        df["WIND_DIR[deg]"] > 360
    ).sum(),

    "GHI < 0": (
        df["GHI[kW1m2]"] < 0
    ).sum(),

    "POA irradiance < 0": (
        df["POA Irr[kW1m2]"] < 0
    ).sum(),

    "Gaia power < 0": (
        df["P_Gaia[kW]"] < 0
    ).sum(),

    "Solar power < 0": (
        df["P_Solar[kW]"] < 0
    ).sum(),

    "Pressure < 800 mbar": (
        df["Pressure[mbar]"] < 800
    ).sum(),

    "Pressure > 1100 mbar": (
        df["Pressure[mbar]"] > 1100
    ).sum(),

    "Elevation < -90°": (
        df["Elevation[deg]"] < -90
    ).sum(),

    "Elevation > 90°": (
        df["Elevation[deg]"] > 90
    ).sum(),
}

range_screening = pd.Series(checks)

print("\nPotentially suspicious observations:")
print(range_screening)

Statistical summary:


,count,mean,std,min,25%,50%,75%,max
TEMPERATURE[degC],658081.0,11.927159,7.213321,-4.980000,5.6850,12.501667,17.500000,44.600000
HUMIDITY[%],658081.0,0.768253,0.156192,0.000000,0.7000,0.800000,0.900000,2.000000
WIND_SPEED[m1s],658081.0,3.588490,2.144343,0.000000,2.0300,3.263333,4.708333,23.922034
WIND_DIR[deg],658081.0,208.943786,73.642975,0.000000,145.7500,215.883333,277.166667,680.000000
GHI[kW1m2],658081.0,0.140380,0.221207,0.000000,0.0000,0.011017,0.203067,1.291350
POA Irr[kW1m2],658081.0,0.144380,0.248869,0.000000,0.0000,0.009000,0.163467,1.187800
P_Gaia[kW],658081.0,2.691753,3.434584,0.000000,0.0000,0.852150,4.734583,23.478949
P_Solar[kW],658081.0,0.988283,1.688230,0.000000,0.0000,0.070645,1.145717,8.404550
Pressure[mbar],658081.0,1011.004293,27.981111,974.215009,1000.0000,1010.881684,1016.700012,2034.500000
Azimuth[deg],658081.0,-2.131793,57.476283,-153.220350,-24.7179,0.000000,17.403900,146.237702



Potentially suspicious observations:
Temperature < -50°C        0
Temperature > 60°C         0
Humidity < 0               0
Wind speed < 0             0
Wind direction < 0°        0
Wind direction > 360°    103
GHI < 0                    0
POA irradiance < 0         0
Gaia power < 0             0
Solar power < 0            0
Pressure < 800 mbar        0
Pressure > 1100 mbar     456
Elevation < -90°           0
Elevation > 90°            0
dtype: int64


In [4]:
# ============================================================
# CELL 4 — CHECK MIDNIGHT ARTIFACT PATTERN
# ============================================================

# We already discovered in the 5-minute dataset that:
#
#   - suspicious pressure values occur at 00:00 UTC
#   - extreme Gaia power values are also strongly
#     concentrated around midnight
#
# Here we only verify whether the same pattern exists
# in the 1-minute dataset.


# ------------------------------------------------------------
# 1. IDENTIFY EXACT MIDNIGHT ROWS
# ------------------------------------------------------------

midnight_mask = (
    (df.index.hour == 0)
    &
    (df.index.minute == 0)
)


# ------------------------------------------------------------
# 2. PRESSURE ABOVE 1100 MBAR
# ------------------------------------------------------------

high_pressure_mask = (
    df["Pressure[mbar]"] > 1100
)

total_high_pressure = high_pressure_mask.sum()

high_pressure_at_midnight = (
    high_pressure_mask
    &
    midnight_mask
).sum()

print("Pressure observations above 1100 mbar:")
print(total_high_pressure)

print("\nPressure >1100 occurring at 00:00 UTC:")
print(high_pressure_at_midnight)


# ------------------------------------------------------------
# 3. GAIA POWER ABOVE 12 KW
# ------------------------------------------------------------

high_gaia_mask = (
    df["P_Gaia[kW]"] > 12
)

total_high_gaia = high_gaia_mask.sum()

high_gaia_at_midnight = (
    high_gaia_mask
    &
    midnight_mask
).sum()

print("\nGaia power observations above 12 kW:")
print(total_high_gaia)

print("\nGaia >12 kW occurring at 00:00 UTC:")
print(high_gaia_at_midnight)


# ------------------------------------------------------------
# 4. MAXIMUM GAIA POWER
# ------------------------------------------------------------

print("\nMaximum Gaia power at midnight:")
print(
    df.loc[
        midnight_mask,
        "P_Gaia[kW]"
    ].max()
)

print("\nMaximum Gaia power outside midnight:")
print(
    df.loc[
        ~midnight_mask,
        "P_Gaia[kW]"
    ].max()
)

Pressure observations above 1100 mbar:
456

Pressure >1100 occurring at 00:00 UTC:
456

Gaia power observations above 12 kW:
541

Gaia >12 kW occurring at 00:00 UTC:
34

Maximum Gaia power at midnight:
23.478949152542373

Maximum Gaia power outside midnight:
13.0486


In [5]:
# ============================================================
# CELL 5 — KEY FORECASTING RELATIONSHIPS
# ============================================================

# We already performed a full correlation investigation
# for the 5-minute dataset.
#
# For the 1-minute dataset, we only need to verify whether
# the most important relationships remain similar.
#
# We focus on:
#
# SOLAR:
#   P_Solar <-> POA irradiance
#   P_Solar <-> GHI
#   P_Solar <-> Elevation
#   P_Solar <-> Humidity
#
# WIND:
#   P_Gaia <-> Wind speed
#
# These values will later be compared across resolutions.


# ------------------------------------------------------------
# SOLAR-POWER CORRELATIONS
# ------------------------------------------------------------

solar_variables = [
    "P_Solar[kW]",
    "POA Irr[kW1m2]",
    "GHI[kW1m2]",
    "Elevation[deg]",
    "HUMIDITY[%]"
]

solar_correlations = (
    df[solar_variables]
    .corr()["P_Solar[kW]"]
    .drop("P_Solar[kW]")
    .sort_values(ascending=False)
)

print("Correlation with P_Solar[kW]:")
print(solar_correlations)


# ------------------------------------------------------------
# WIND-POWER CORRELATION
# ------------------------------------------------------------

wind_power_correlation = df[
    [
        "P_Gaia[kW]",
        "WIND_SPEED[m1s]"
    ]
].corr().loc[
    "P_Gaia[kW]",
    "WIND_SPEED[m1s]"
]

print("\nCorrelation between P_Gaia and wind speed:")
print(wind_power_correlation)

Correlation with P_Solar[kW]:
POA Irr[kW1m2]    0.995232
GHI[kW1m2]        0.913345
Elevation[deg]    0.743143
HUMIDITY[%]      -0.599584
Name: P_Solar[kW], dtype: float64

Correlation between P_Gaia and wind speed:
0.8167945778774939


## 1-Minute EDA — Main Findings

The exploratory analysis of the 1-minute SOLETE dataset was kept
intentionally compact because the 60-minute and 5-minute resolutions
had already been investigated in greater detail.

### Dataset Structure

- The dataset contains **658,081 observations** and **11 measured variables**.
- Temporal coverage extends from **June 2018 to September 2019**.
- No missing values were found.
- No duplicate timestamps were found.
- All timestamps are already stored in chronological order.
- Every consecutive observation is separated by exactly **1 minute**.
- No unexpected sampling intervals were detected.

### Data-Quality Observations

The 1-minute dataset preserves several data-quality patterns already
identified at other SOLETE resolutions.

- **103 wind-direction observations** exceed 360 degrees.
- **456 pressure observations** exceed 1100 mbar.
- All 456 unusually high pressure observations occur exactly at
  **00:00 UTC**.
- This confirms that the systematic midnight pressure artifact is also
  present at 1-minute resolution.

Wind direction is a circular variable and should not be treated as an
ordinary linear feature during modelling. Circular representations such
as sine and cosine will be considered during feature engineering.

### Wind-Power Behaviour

The relationship between Gaia turbine power and wind speed remains
strong:

- `P_Gaia[kW]` vs `WIND_SPEED[m1s]`: approximately **0.817**

The 1-minute dataset contains:

- **541 observations** with `P_Gaia[kW] > 12`
- only **34** of these occur exactly at midnight
- maximum Gaia power at midnight: approximately **23.48 kW**
- maximum Gaia power outside midnight: approximately **13.05 kW**

Therefore, values slightly above 12 kW should **not automatically be
classified as invalid** at 1-minute resolution.

The much larger midnight extremes remain suspicious because they occur
at the same daily boundary where other systematic artifacts are present.

The somewhat weaker wind-speed/power correlation compared with the
5-minute dataset may reflect additional short-term variability retained
at the finer temporal resolution.

### Solar-Power Relationships

The major solar relationships are highly consistent with those observed
in the 5-minute dataset.

Correlation with `P_Solar[kW]`:

- `POA Irr[kW1m2]`: approximately **0.995**
- `GHI[kW1m2]`: approximately **0.913**
- `Elevation[deg]`: approximately **0.743**
- `HUMIDITY[%]`: approximately **-0.600**

This consistency suggests that the main physical relationships between
solar generation, irradiance, and solar geometry are preserved across
the 1-minute and 5-minute resolutions.

### Correlation Interpretation Note

The correlations calculated above describe variables observed at the
**same timestamp** and should not be interpreted directly as forecasting
performance.

In particular:

- solar power, GHI, and POA irradiance contain many simultaneous low or
  zero nighttime observations, which can strengthen whole-dataset
  Pearson correlations;
- contemporaneous `GHI` and `POA Irr` values may not be available for
  the future timestamp being predicted;
- using future measurements that would not realistically be known at
  forecast time could introduce **information leakage**.

The forecasting experiments will therefore distinguish between:

1. variables available at the forecast origin,
2. lagged historical measurements,
3. deterministic future features such as calendar and astronomical
   information,
4. and variables whose future values would themselves require forecasts.

### Humidity Encoding Note

The observed values of `HUMIDITY[%]` do not resemble conventional
percentage-point relative humidity values on a 0–100 scale.

The original values are therefore preserved without transformation.

The exact encoding or unit of this variable should be verified before
humidity is used in the final forecasting pipeline.

### Cross-Resolution Observation

The 1-minute and 5-minute datasets preserve very similar solar-power
relationships.

The finer 1-minute resolution retains more short-term wind-power
variability, while the systematic midnight pressure anomaly is present
at both resolutions.

This indicates that the pressure issue is not simply introduced by
5-minute aggregation.

### Data-Handling Decision

No source observations were modified during EDA.

Potential preprocessing decisions will be handled explicitly during the
modelling stage, including:

- treatment or exclusion of anomalous pressure values,
- circular encoding of wind direction,
- verification of humidity encoding,
- handling of suspicious midnight observations,
- and realistic feature-availability rules to prevent information
  leakage.

### Overall 1-Minute Assessment

The 1-minute dataset is structurally clean and preserves the important
solar and wind relationships observed at 5-minute resolution.

Its main concerns are related to data representation and systematic
daily-boundary artifacts rather than missing data or timestamp
irregularities.

### Next Step

The 1-second SOLETE dataset will be examined using a lightweight and
memory-conscious EDA.

Afterward, a dedicated cross-resolution analysis will compare the
60-minute, 5-minute, 1-minute, and 1-second datasets before selecting
the main temporal resolution for forecasting experiments.